In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_json, struct

spark = SparkSession.builder \
    .appName("KafkaGlobalProducer") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

data = [
    (101, "Kenan Production Grid is fully alive!"),
    (102, "Real-time streaming via plug-and-play architecture.")
]
columns = ["id", "message"]
df = spark.createDataFrame(data, columns)

kafka_df = df.select(to_json(struct(col("id"), col("message"))).alias("value"))

kafka_df.write \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic", "cluster_kafka_topic") \
    .save()

print("Topic is There !!")

In [ ]:
streaming_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "cluster_kafka_topic") \
    .option("startingOffsets", "earliest") \
    .load()

clean_stream_df = streaming_df.selectExpr("CAST(value AS STRING) as payload")

query = clean_stream_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()

query.awaitTermination(timeout=20)
print("see your console in Docker")